# 10 · Neural Network & Foundation Model Tabular (dijalankan di **Kaggle GPU**)

Tiga model dengan cara kerja yang sangat berbeda dari GBDT, untuk menambah keberagaman ensemble:

| Model | Cara kerja | Kenapa cocok untuk data kita |
|---|---|---|
| **RealMLP** (pytabkit) | MLP dengan default yang sudah dituning: robust scaling + *smooth clipping*, embedding numerik periodik, label smoothing | hubungan fitur → risiko di data ini halus & aditif. MLP memodelkan fungsi halus secara langsung, sedangkan GBDT memotongnya jadi tangga |
| **TabM** (pytabkit) | satu jaringan meniru ensemble 32 MLP (bobot sebagian besar dibagi) + embedding numerik *piecewise-linear* | MLP terkuat di benchmark TabArena. Ensemble internalnya meredam noise |
| **TabICLv2** | *foundation model* yang sudah dilatih awal: "membaca" data train sebagai konteks lalu langsung memprediksi, **tanpa training** | prior yang dipelajari dari jutaan dataset sintetis, jadi sangat berbeda dari semua model kita |

**Fold identik** dengan LightGBM (dari `folds.parquet`), sehingga OOF bisa digabung berpasangan secara lokal di notebook 11.

**Pengaturan notebook Kaggle:** *Accelerator* = GPU (T4 x2 atau P100), *Internet* = On, *Input* = dataset paket dari notebook 09.

> Setiap model langsung disimpan setelah selesai. Kalau sesi terputus, jalankan ulang: model yang file run-nya sudah ada dilewati (`SKIP_DONE = True`). Waktu per fold dicetak, jadi Anda bisa memperkirakan sisa waktu.

In [ ]:
# Instal paket (butuh Internet = On). PyTorch & CUDA sudah tersedia di image Kaggle.
!pip install -q "polars>=1.30" pytabkit tabm rtdl_num_embeddings tabicl 2>&1 | tail -2

In [ ]:
import glob
import json
import os
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch

# cari folder paket: di Kaggle (/kaggle/input/...) atau lokal (../kaggle_bundle)
cands = glob.glob("/kaggle/input/**/folds.parquet", recursive=True) + glob.glob("../kaggle_bundle/folds.parquet")
assert cands, "Paket tidak ditemukan — tambahkan dataset paket (notebook 09) lewat 'Add Input'"
BUNDLE = Path(cands[0]).parent
sys.path.insert(0, str(BUNDLE))
import modeling as M
import nn_models as N

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Paket:", BUNDLE, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (akan sangat lambat)")
print("Run disimpan ke:", M.PRED_DIR)

meta = json.loads((BUNDLE / "features_v1.json").read_text())
train = pl.read_parquet(BUNDLE / "train_v1.parquet")
test = pl.read_parquet(BUNDLE / "test_v1.parquet")
folds = pl.read_parquet(BUNDLE / "folds.parquet")
FEAT, CATS = meta["features"], meta["categorical"]
y = train["target"].to_numpy()
print(f"train {train.shape} | test {test.shape} | {len(FEAT)} fitur | e-commerce: {meta['ecommerce_source']}")

## Konfigurasi

- **RealMLP & TabM:** 3 seed × 5 fold, seed yang sama dengan anggota ensemble lain (42, 7, 2024). Validasi fold dipakai untuk memilih epoch terbaik (AUC), setara *early stopping* di LightGBM.
- **TabM** memakai embedding numerik *piecewise-linear* (`num_emb_type="pwl"`), varian terbaik di paper TabM. Default pytabkit tanpa embedding.
- **TabICL:** mulai dengan 1 seed (5 fold). Model ini tidak dilatih; tiap fold "membaca" 56.000 baris sebagai konteks. Kalau cepat, tambahkan seed 7 & 2024 supaya bisa dibandingkan per seed.

Hapus atau komentari baris model yang tidak ingin dijalankan.

In [ ]:
RUN_CONFIG = {
    "realmlp_v1": dict(kind="realmlp", seeds=(42, 7, 2024), params={}),
    "tabm_v1":    dict(kind="tabm",    seeds=(42, 7, 2024), params=dict(num_emb_type="pwl")),   # + embedding piecewise-linear
    "tabicl_v1":  dict(kind="tabicl",  seeds=(42,),         params=dict(n_estimators=8)),
}
SKIP_DONE = True

In [ ]:
RESULTS = {}
for name, cfg in RUN_CONFIG.items():
    if SKIP_DONE and (M.PRED_DIR / f"{name}_meta.json").exists():
        print(f"{name}: sudah ada, dilewati")
        continue
    print(f"\n===== {name} =====")
    t = time.time()
    res = N.train_nn_cv(cfg["kind"], train, test, FEAT, CATS, folds, seeds=cfg["seeds"], params=cfg["params"], device=DEVICE)
    M.save_run(name, train, test, res, extra=dict(kind=cfg["kind"], feature_version="v1",
                                                  minutes=round((time.time() - t) / 60, 1)))
    RESULTS[name] = res
    print(f"{name}: tersimpan ({(time.time() - t) / 60:.1f} menit)")
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## Cek cepat terhadap `lgbm_tuned`

Ini hanya pratinjau: korelasi peringkat dan rata-rata peringkat sederhana dengan LightGBM, kalau file `lgbm_tuned` ikut di paket. Keputusan ensemble yang sebenarnya diambil secara lokal di notebook 11, bersama semua anggota lain.

In [ ]:
if (BUNDLE / "lgbm_tuned_oof.parquet").exists():
    import shutil as _sh
    for ext in ("oof", "test", "meta"):
        f = BUNDLE / f"lgbm_tuned_{ext}.{'json' if ext == 'meta' else 'parquet'}"
        _sh.copy(f, M.PRED_DIR / f.name) if not (M.PRED_DIR / f.name).exists() else None
    base = M.run_from_disk("lgbm_tuned", y)
    rows = []
    for name in RUN_CONFIG:
        if not (M.PRED_DIR / f"{name}_meta.json").exists():
            continue
        r = M.run_from_disk(name, y)
        s = [s for s in r["seeds"] if s in base["seeds"]]
        corr = np.corrcoef(M.rank01(r["oof_by_seed"][s[0]]), M.rank01(base["oof_by_seed"][s[0]]))[0, 1]
        b = M.blend({"lgbm": base, name: r}, y, seeds=s)
        cmp = M.compare_runs(base, b, y)
        rows.append(dict(model=name, ks_oof=r["ks_oof"], ks_lgbm_seed_sama=M.ks_score(y, np.mean([base["oof_by_seed"][x] for x in s], axis=0)),
                         korelasi_peringkat=corr, delta_blend_50_50=cmp["delta_oof"], seed=str(s)))
    display(pl.DataFrame(rows).with_columns(pl.col(pl.Float64).round(4)))

In [ ]:
# kemas semua run untuk diunduh (tab Output → nn_runs.zip), lalu ekstrak ke data/preds/ lokal
names = [p.name for p in M.PRED_DIR.iterdir() if any(p.name.startswith(n) for n in RUN_CONFIG)]
tmp = Path("/kaggle/working/nn_runs") if Path("/kaggle/working").exists() else M.ROOT / "nn_runs"
tmp.mkdir(parents=True, exist_ok=True)
for n in names:
    shutil.copy(M.PRED_DIR / n, tmp / n)
zip_path = shutil.make_archive(str(tmp), "zip", tmp)
print("Siap diunduh:", zip_path)
print("\n".join(sorted(names)))